In [6]:
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import kagglehub

sns.set_theme()
path = kagglehub.dataset_download("yeanzc/telco-customer-churn-ibm-dataset")

df = pd.read_excel(path + "/telco_customer_churn.xlsx")

target = df["Churn Value"]


print(df.head(10))

   CustomerID  Count        Country       State         City  Zip Code  \
0  3668-QPYBK      1  United States  California  Los Angeles     90003   
1  9237-HQITU      1  United States  California  Los Angeles     90005   
2  9305-CDSKC      1  United States  California  Los Angeles     90006   
3  7892-POOKP      1  United States  California  Los Angeles     90010   
4  0280-XJGEX      1  United States  California  Los Angeles     90015   
5  4190-MFLUW      1  United States  California  Los Angeles     90020   
6  8779-QRDMV      1  United States  California  Los Angeles     90022   
7  1066-JKSGK      1  United States  California  Los Angeles     90024   
8  6467-CHFZW      1  United States  California  Los Angeles     90028   
9  8665-UTDHZ      1  United States  California  Los Angeles     90029   

                 Lat Long   Latitude   Longitude  Gender  ...        Contract  \
0  33.964131, -118.272783  33.964131 -118.272783    Male  ...  Month-to-month   
1   34.059281, -118.307

In [11]:
#limpando dados

df["Total Charges"] = pd.to_numeric(df["Total Charges"], errors="coerce")
df['Total Charges'] = df['Total Charges'].fillna(0)

col_to_drop = ["Count", "State", "Country","CustomerID", "Lat Long", "Churn Reason", "Churn Label", "City", "Gender",
    "Senior Citizen",
    "Partner",
    "Dependents",
    "Phone Service",
    "Multiple Lines",
    "Internet Service",
    "Online Security",
    "Online Backup",
    "Device Protection",
    "Tech Support",
    "Streaming TV",
    "Streaming Movies",
    "Contract",
    "Paperless Billing",
    "Payment Method"]



df_cleaned = df.drop(columns=col_to_drop, errors='ignore')


In [12]:
model_configs = {
    "Logistic Regression": {
        "model_key": "logistic_regression",
        "model_params": {
            "random_state": 42,
            "max_iter": 1000,
        },
        "preprocessing": {
            "scaling": "standard",
        },
        "artifact_name": "logistic_regression",
    },
    "Random Forest": {
        "model_key": "random_forest",
        "model_params": {
            "n_estimators": 200,
            "max_depth": 12,
            "random_state": 42,
        },
        "preprocessing": {
            "scaling": "none",
        },
        "artifact_name": "random_forest",
    },
    "Gradient Boosting": {
        "model_key": "gradient_boosting",
        "model_params": {
            "random_state": 42,
        },
        "preprocessing": {
            "scaling": "none",
        },
        "artifact_name": "gradient_boosting",
    },
    "SVC": {
        "model_key": "svc",
        "model_params": {
            "kernel": "rbf",
            "C": 1.0,
            "random_state": 42,
        },
        "preprocessing": {
            "scaling": "standard",
        },
        "artifact_name": "svc",
    },
    "KNN": {
        "model_key": "knn",
        "model_params": {
            "n_neighbors": 7,
        },
        "preprocessing": {
            "scaling": "standard",
        },
        "artifact_name": "knn",
    },
}


In [13]:
# testando baselines pra ver qual melhor modelo usar depois e separando dados de treino e teste
from sklearn.model_selection import train_test_split


X = df_cleaned.drop(columns=["Churn Value"], errors='ignore')
y = df_cleaned["Churn Value"]

x_train, x_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [14]:
# importando modelos de baseline
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.feature_selection import SelectKBest, f_classif

scaler = StandardScaler()
mlflow.set_experiment("customer_churn_ibm")
selector = SelectKBest(score_func=f_classif, k=min(10, x_train.shape[1]))
x_train_selected = selector.fit_transform(x_train, y_train)
x_test_selected = selector.transform(x_test)

model_registry = {
    "logistic_regression": LogisticRegression,
    "random_forest": RandomForestClassifier,
    "gradient_boosting": GradientBoostingClassifier,
    "svc": SVC,
    "knn": KNeighborsClassifier,
}


In [ ]:
# comparando diferentes algoritmos de classificacao
comparison_rows = []
best_model = None
best_model_name = None
best_model_score = -1.0
best_model_artifact_name = None

for model_name, config in model_configs.items():
    model_class = model_registry[config["model_key"]]
    model = model_class(**config["model_params"])

    if config["preprocessing"]["scaling"] == "standard":
        x_train_prepared = scaler.fit_transform(x_train_selected)
        x_test_prepared = scaler.transform(x_test_selected)
    else:
        x_train_prepared = x_train_selected
        x_test_prepared = x_test_selected

    with mlflow.start_run(run_name=model_name):
        model.fit(x_train_prepared, y_train)

        y_pred = model.predict(x_test_prepared)
        train_accuracy = accuracy_score(y_train, model.predict(x_train_prepared))
        accuracy = accuracy_score(y_test, y_pred)
        precision = precision_score(y_test, y_pred, zero_division=0)
        recall = recall_score(y_test, y_pred, zero_division=0)
        f1 = f1_score(y_test, y_pred, zero_division=0)
        overfitting = train_accuracy - accuracy

        mlflow.log_param("model_key", config["model_key"])
        mlflow.log_param("scaling", config["preprocessing"]["scaling"])
        mlflow.log_metric("accuracy", accuracy)
        mlflow.log_metric("precision", precision)
        mlflow.log_metric("recall", recall)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("overfitting", overfitting)
        mlflow.sklearn.log_model(model, config["artifact_name"])

        comparison_rows.append(
            {
                "model": model_name,
                "accuracy": accuracy,
                "precision": precision,
                "recall": recall,
                "f1_score": f1,
                "overfitting": overfitting,
            }
        )

        if f1 > best_model_score:
            best_model_score = f1
            best_model = model
            best_model_name = model_name
            best_model_artifact_name = config["artifact_name"]

comparison_results = pd.DataFrame(comparison_rows).sort_values(
    by=["f1_score", "accuracy"], ascending=False
).reset_index(drop=True)

print(comparison_results.to_string(index=False))
print(
    f"\nMelhor modelo por F1 Score: {best_model_name} - F1 Score: {best_model_score:.4f}"
)


2026/05/27 19:24:54 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


In [ ]:
# persistindo o melhor modelo comparado
import joblib

best_model_filename = f"{best_model_artifact_name}.pkl"
mlflow.sklearn.log_model(best_model, best_model_artifact_name)
joblib.dump(best_model, best_model_filename)

print(f"Modelo salvo em {best_model_filename}")


2026/05/19 17:49:09 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


['logistic_regression_baseline.pkl']